# Exercise 7 Minimal

This notebook imports the modularized Werewolf game package from the local `Agents/` directory.

In [1]:
%load_ext autoreload
%reload_ext autoreload
%autoreload 2

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from Agents.state import DayChannel, DayVote, InvestigatorResult, OrchestratorGraph, WolfChannel
from Agents.prompts import GAME_PREAMBLE
from Agents.schemas import DayDiscussOutput, DayVoteOutput, HealerOutput, InvestigatorOutput, WolfNightDiscussOutput
from Agents.formatters import format_day_channel, format_investigator_results, format_wolf_channel
from Agents.agents import (
    healer_act,
    investigator_act,
    investigator_discuss,
    investigator_vote,
    villager_discuss,
    villager_vote,
    wolf_discuss,
    wolf_night_discuss,
    wolf_vote,
)
from Agents.nodes import initialize_game, day_resolution, night_resolution, end_game
from Agents.graphs.day import day_graph_compiled
from Agents.graphs.wolf_night import wolf_night_graph_compiled
from Agents.graphs.healer import healer_graph_compiled
from Agents.graphs.investigator import investigator_graph_compiled
from Agents.graphs.parent import parent_graph_compiled
from Agents.main import run_game

In [4]:
# Requires GOOGLE_API_KEY in your environment or .env file.
from Agents.agents import prompt_log
from tests.leak_test import run_leak_tests

result = run_game()
leaks = run_leak_tests(prompt_log, result["roles"])
assert not leaks

print(f"Winner: {result['winner']}")
print(f"Game lasted {result['current_day']} days")
print(f"Surviving wolves: {result['surviving_wolves']}")
print(f"Surviving villagers: {result['surviving_villagers']}")

=== Running Leak Tests ===
=== Leak Tests Complete ===
Winner: villagers
Game lasted 3 days
Surviving wolves: []
Surviving villagers: ['player_1', 'player_4', 'player_5', 'player_6', 'player_8']


In [5]:
from Agents.memory import store

for role in ["wolf", "villager", "healer", "investigator"]:
    print(role, store.get(("strategy", role), "latest"))



wolf Item(namespace=['strategy', 'wolf'], key='latest', value={'game_id': 'game_20260508_130942', 'content': 'Your survival depends on blending in through both communication and voting. When questioned, avoid deflection and evasiveness at all costs; it is better to offer a plausible, generic theory than to turn questions back on your accusers. This pattern of behavior is a massive red flag for villagers. Crucially, if your wolf partner is about to be voted out by a strong majority, vote with the village to eliminate them. Casting a lone dissenting vote creates an undeniable link on the public record, making you the next logical target. Sacrificing your partner is often the correct move to ensure your own survival.', 'created_at': datetime.datetime(2026, 5, 8, 13, 11, 10, 566625)}, created_at='2026-05-08T05:11:11.407890+00:00', updated_at='2026-05-08T05:11:11.407896+00:00')
villager Item(namespace=['strategy', 'villager'], key='latest', value={'game_id': 'game_20260508_130942', 'content

In [6]:

for role in ["wolf", "villager", "healer", "investigator"]:
    print(store.search(("observations", role), limit=10))

[Item(namespace=['observations', 'wolf'], key='571bb196-a1cd-49c1-91a7-67469304aa0e', value={'observation_count': 2, 'last_observed': datetime.datetime(2026, 5, 8, 13, 11, 2, 690505), 'game_id': '1', 'content': 'Scenario: Two wolves are voting on a day when the village has a strong consensus on a different target. Approach: The two wolves both voted for a third, less suspicious player instead of voting with the majority to eliminate the main target. Outcome: After one wolf was eliminated, villagers analyzed the voting record and identified the other wolf because they had voted in alignment with a known enemy, leading to their elimination the next day.'}, created_at='2026-05-08T05:11:03.074944+00:00', updated_at='2026-05-08T05:11:03.074954+00:00', score=None), Item(namespace=['observations', 'wolf'], key='4b9a4ddc-32ef-4c12-8910-f06e13142b5d', value={'observation_count': 1, 'last_observed': datetime.datetime(2026, 5, 8, 13, 2, 22, 702044), 'game_id': '1', 'content': "Scenario: A lone wo

In [7]:

for role in ["wolf", "villager", "healer", "investigator"]:
    print(store.search(("strategy_points", role), limit=10))

[Item(namespace=['strategy_points', 'wolf'], key='6235ef0c-78b4-4f17-915e-4bbc50cc7b12', value={'observation_count': 2, 'last_observed': datetime.datetime(2026, 5, 8, 13, 11, 6, 40835), 'game_id': 'game_20260508_110628', 'content': 'When your wolf partner is under heavy suspicion and a strong village consensus has formed against them, making their elimination almost certain.', 'action': 'Do not cast a lone dissenting vote for a villager when the rest of the village is correctly voting out your wolf partner. Vote with the majority to eliminate your partner; this prevents you from being immediately implicated by the voting record after their role is revealed.'}, created_at='2026-05-08T05:11:06.384861+00:00', updated_at='2026-05-08T05:11:06.384871+00:00', score=None), Item(namespace=['strategy_points', 'wolf'], key='3dc04eb6-00fa-4a7a-bb47-51f770d28afa', value={'observation_count': 1, 'last_observed': datetime.datetime(2026, 5, 8, 13, 3, 57, 775292), 'game_id': 'game_20260508_110628', 'co

In [8]:
for msg in result["day_channel"]:
    print(f"[Day {msg.day}, Round {msg.round}] {msg.player}: {msg.message}")

[Day 1, Round 1] player_1: Alright everyone, Day 1! Who wants to kick us off with their initial thoughts? Any first suspicions or anything we should discuss right away?
[Day 1, Round 1] player_3: Alright everyone, Day 1! Who wants to kick us off with their initial thoughts? Let's get some discussion going.
[Day 1, Round 1] player_4: Alright team, Day 1! Who's got some initial thoughts? Any first impressions?
[Day 1, Round 1] player_5: Alright everyone, let's get this discussion started! Any initial thoughts or suspicions from anyone?
[Day 1, Round 1] player_6: Alright everyone, Day 1! Who wants to kick things off? Any initial thoughts or weird vibes?
[Day 1, Round 1] player_8: Alright, Day 1! Who wants to kick us off with their initial thoughts or suspicions? Any early reads on anyone?
[Day 1, Round 1] player_2: Alright, Day 1! Anyone have any initial thoughts or theories? It's always tough at the start.
[Day 1, Round 1] player_7: Alright, day 1! Anyone have any initial thoughts or any

# Long term memory implementation

In [9]:
prompt_log

[{'player_id': 'player_8',
  'player_role': 'healer',
  'output_key': 'day_channel',
  'day': 1,
  'round': 1,
  'prompt_input': {'player_id': 'player_8',
   'player_role': 'healer',
   'current_day': 1,
   'current_round': 1,
   'surviving_players': 'player_1, player_3, player_4, player_5, player_6, player_8, player_2, player_7',
   'surviving_wolves': '',
   'surviving_villagers': '',
   'day_channel': 'No messages yet.',
   'day_summaries': 'No previous day summaries yet.',
   'wolf_channel': 'No messages yet.',
   'investigator_results': 'No investigations yet.',
   'previous_strategy': '',
   'strategy_points': '',
   'retrieved_observations': 'No past observations available.'}},
 {'player_id': 'player_5',
  'player_role': 'villager',
  'output_key': 'day_channel',
  'day': 1,
  'round': 1,
  'prompt_input': {'player_id': 'player_5',
   'player_role': 'villager',
   'current_day': 1,
   'current_round': 1,
   'surviving_players': 'player_1, player_3, player_4, player_5, player_6, 